#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window


# The Transformation Logic

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
    ci.customer_id,
    ci.firstname,
    ci.lastname,
    la.country,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender
        ELSE COALESCE(ca.gender, 'n/a')
    END AS gender,
    ca.birth_date AS birthdate,
    ci.created_date AS create_date
FROM workspace_1.silver.crm_customers ci
LEFT JOIN workspace_1.silver.erp_customers_id ca
    ON ci.customer_key = ca.customer_number
LEFT JOIN workspace_1.silver.erp_customer_location la
    ON ci.customer_key = la.customer_number
"""
df = spark.sql(query)

In [0]:
df.limit(10).display()

# Write It To Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace_1.gold.dim_customers")

#Sanity Checks Gold Tables

In [0]:
%sql
SELECT * FROM workspace_1.gold.dim_customers LIMIT 10